In [ ]:
from gp import GaussianProcess
import pandas as pd
import jax
import jax.numpy as jnp

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
sequoia = pd.read_csv('../data/sequoia.csv')

In [ ]:
sequoia.head()

In [ ]:
sequoia_nonan = sequoia[sequoia.signal_measured]

scaler = MinMaxScaler()
X_train = pd.DataFrame()
X_train[['latitude', 'longitude']] = scaler.fit_transform(sequoia_nonan[['latitude', 'longitude']])
X_train['indoor'] = sequoia_nonan['indoor'].astype(float).values

y_train = sequoia_nonan['rssi']

X_train = jnp.array(X_train)
y_train = jnp.array(y_train)

In [ ]:
def m(obs):
    return sequoia.rssi.mean()

bandwidth = 10
def K(obs1, obs2):
    return jnp.exp(-jnp.linalg.norm(obs1 - obs2, ord=2) / (2*bandwidth))

In [ ]:
GP = GaussianProcess(m, K)

In [ ]:
GP.fit(X_train, y_train)

In [ ]:
grid_size = 50

x_new = jnp.linspace(0, 1, grid_size)
y_new = jnp.linspace(0, 1, grid_size)
grid_points = jnp.stack(jnp.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)
coord_grid = scaler.inverse_transform(grid_points)
long_new, lat_new = coord_grid.T

# match indoor to flag of closest observed point
from scipy.spatial import cKDTree
tree = cKDTree(sequoia[["latitude", "longitude"]].to_numpy())
_, indices = tree.query(coord_grid)
z_new = sequoia["indoor"].to_numpy()[indices].reshape(-1,1)


# z_new = jnp.ones([grid_size**2, 1]) * 0

X_new = jnp.concatenate([grid_points, z_new], axis=-1)

In [ ]:
candidates=jnp.array([0, 1]).reshape(-1,1)
X_best, y_mean, y_cov = GP.predict(grid_points, candidates=candidates)

from wifiplotting import plot_wifi_heatmap, lonlat_to_world
df_new = pd.DataFrame(X_new, columns=['latitude', 'longitude', 'indoor'])
plot_wifi_heatmap(df_new, new_data=y_mean, aggregate=False)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.scatter(lat_new, long_new, c=y_mean, cmap="RdYlGn")
plt.colorbar()
plt.ticklabel_format(style='plain', axis='both', useOffset=False)
plt.tight_layout()

In [ ]:
print(f"Maximum RSSI Coordinates: ({long_new[y_mean.argmax()]}, {lat_new[y_mean.argmax()]}) ")

In [ ]:
fig, ax, rssi_points, na_points, zoom = plot_wifi_heatmap(sequoia, show_na=0.5)
rows_used = int(rssi_points["sample_count"].sum() + na_points["sample_count"].sum())
print(f"RSSI heatmap uses {len(rssi_points) + len(na_points)} unique coordinate bins from {rows_used} rows.")

# doesn't work
# xs, ys = zip(*[lonlat_to_world(lon, lat, zoom) for lon, lat in zip(long_new, lat_new)])
# ax.scatter(xs, ys, c=y_mean, cmap="RdYlGn")

plt.show()